# TMM vs Paper — 文献图复现

用 TMM 复现教材/论文中的经典算例，与 [`03_example_vs_virtuallab.ipynb`](03_example_vs_virtuallab.ipynb)（商业软件对照）并列。

**运行前提**：在 `simulation_core` 根目录执行 `source scripts/init-simulation-build-env.sh build`，再 `./assets/ipynb/simulation/TMM/run_tmm.sh jupyter`（或在已 source 的环境中打开本 notebook）。须具备 `SIMULATION_ARTIFACTS_DIR`（Release `build/`）与 `SIMULATION_DATABASE_DIR`（YAML `assets/database`）；**勿**使用 `init-toykits-build-env.sh` / `.simulation_toolkits`。


In [1]:
import numpy as np
from simulation import *


def make_layer(nk, depth, name=""):
    return make_layer_from_nk_s(complex(nk), float(depth), name)

def layer_nk(lyr, wl_um):
    return lyr.background_material.nk_at_wavelength_um(float(wl_um))
from math import *
import matplotlib.pyplot as plt
from viz_io import save_or_show_fig
from IPython.display import Image

degree = pi / 180


def resolve_layers(layers, wl_um):
    return TMM_resolve_nk_depth_at_wl_s(layers, float(wl_um))


def print_layers(layers, wl_um=633.0):
    for i, lyr in enumerate(layers):
        nk = lyr.background_material.nk_at_wavelength_um(float(wl_um))
        print(f"  layer {i}: depth={lyr.depth} nm, nk={nk}")


## 椭圆偏振 — Tompkins Fig. 1.14

膜系 `air | SiO₂(d) | Si`，633 nm，70° 入射角。计算 Ψ（振幅比）和 Δ（相位差）随 SiO₂ 厚度（0–1000 nm）的变化。

文献对照：*Handbook of Ellipsometry* (Tompkins, 2005) Fig. 1.14。图中 Ψ 为蓝线，Δ 为红线。


In [ ]:
def ellips(layers, crao, wavelength):
    meterials = resolve_layers(layers, wavelength)
    dir = TMM_propagate_direction_s(meterials, crao, wavelength)
    rs, ts = TMM_get_r_t_from_tmm(TMM_interface_transfer_matrix_with_thickness_s(meterials, dir, wavelength)[-1])
    rp, tp = TMM_get_r_t_from_tmm(TMM_interface_transfer_matrix_with_thickness_p(meterials, dir, wavelength)[-1])
    return np.arctan(abs(rp/rs)), np.angle(-rp/rs)

def ellipsometric_sample():
    """
    Here is a calculation of the psi and Delta parameters measured in
    ellipsometry. This reproduces Fig. 1.14 in Handbook of Ellipsometry by
    Tompkins, 2005.
    """
    
    air = make_layer_from_nk_s(1.0 + 0j, 0.0, "air")
    sio2 = make_layer_from_nk_s(1.46 + 0j, 0.0, "sio2")
    si = make_layer_from_nk_s(3.87 + 0.02j, 1.0e6, "si")

    ds = np.linspace(0, 1000, num=100) #in nm
    psis = []
    Deltas = []
    degree = pi/180

    for d in ds:
        sio2.depth = d
        psi, delta_phase = ellips([air, sio2, si], 70*degree, 633) #in nm
        psis.append(psi/degree) # angle in degrees
        Deltas.append(delta_phase/degree) # angle in degrees
    fig = plt.figure()
    plt.plot(ds, psis, 'blue', label=r'$\Psi$')
    plt.plot(ds, Deltas, 'red', label=r'$\Delta$')
    plt.xlabel('SiO2 thickness (nm)')
    plt.ylabel('Ellipsometric angles (degrees)')
    plt.title('Ellipsometric parameters for air/SiO2/Si, varying '
            'SiO2 thickness @ 70°, 633 nm. '
            'Compare with Handbook of Ellipsometry Fig. 1.14')
    plt.legend()
    save_or_show_fig(fig, title='Ellipsometric_parameters_SiO2_Si')

print("* 计算在不同厚度下, 椭圆偏振的振幅比和相位差")
ellipsometric_sample()

In [ ]:
Image(filename='../resource/tmm/ellipsometric_sample.png')

## 表面等离子体共振 (SPR) — Mater. Trans. Fig. 6a

膜系 `BK7 | Cr(5 nm) | Au(30 nm) | air`，633 nm，p 偏振。反射/透射率随入射角（30°–60°）变化，应在 ~43° 附近出现反射率 dip。

文献对照：[Mater. Trans. M2010003](http://doi.org/10.2320/matertrans.M2010003) Fig. 6a。


In [ ]:
def reflection_of_surface_plasmon_resonance():

    layers = [
        make_layer_from_nk_s(1.517 + 0j, 0.0, "bk7"),
        make_layer_from_nk_s(3.719 + 4.362j, 5.0, "chromium"),
        make_layer_from_nk_s(0.130 + 3.162j, 30.0, "gold"),
        make_layer_from_nk_s(1.0 + 0j, 0.0, "air"),
    ]
    lam_vac = 633
    print_layers(layers, lam_vac)
    print(f"wavelength={lam_vac}nm")
    meterials = resolve_layers(layers, lam_vac)
    theta_list = np.linspace(30*degree, 60*degree, num=300)
    Rp = []
    Tp=[]
    for theta in theta_list:
        dir = TMM_propagate_direction_s(meterials, theta, lam_vac)
        dir[0] = TMM_apply_surface_plasmon_resonance_effect_s(layer_nk(meterials[0], lam_vac), dir[0])
        dir[-1] = TMM_apply_surface_plasmon_resonance_effect_s(layer_nk(meterials[-1], lam_vac), dir[-1])
        rp, tp = TMM_get_r_t_power_from_tmm_p(TMM_interface_transfer_matrix_with_thickness_p(meterials, dir, lam_vac)[-1],
            layer_nk(meterials[0], lam_vac), dir[0], layer_nk(meterials[-1], lam_vac), dir[-1]
        )
        Rp.append(rp)
        Tp.append(tp)
    fig = plt.figure()
    plt.plot(theta_list/degree, Rp, 'blue', label='R_p')
    plt.plot(theta_list/degree, Tp, 'red', label='T_p')
    plt.xlabel('theta (degree)')
    plt.ylabel('R / T')
    plt.xlim(30, 60)
    plt.ylim(0, 1)
    plt.title('p-polarized R and T with Surface Plasmon Resonance\n'
              'Compare with http://doi.org/10.2320/matertrans.M2010003 Fig 6a'
    )
    plt.legend()
    save_or_show_fig(fig, title='SPR_p_polarized_RT')


print(
    "*  SPR现象早在1902年由Wood发现，他观察到当电磁波射向金属表面时，\n"
    "其反射光谱会产生异常，表现为在特定角度下反射光强度明显下降，\n"
    "光谱上出现明显的暗带。而且金属膜表面折射率的变大会导致暗带位置发生变化\n"
)
reflection_of_surface_plasmon_resonance()


In [ ]:
Image(filename='../resource/tmm/reflection_of_surface_plasmon_resonance.png')